# Lab 2 — The fanout complaint reaches the residents’ meeting

BI joins order headers to line items. Finance spots its order total repeated once per line and calls an emergency meeting. Nobody has to be malicious: the query multiplied a higher-grain fact across lower-grain rows. This lab makes the multiplication visible, then uses Metric View cardinality to model two fact sources at their own grains.

This bundle writes to `workspace.metric_views_lab` by default; change that catalog/schema in the SQL cells if needed. If the schema does not exist, the schema-creation cell requires `CREATE SCHEMA` on the parent catalog.

Requires Unity Catalog, access to `samples.tpch.orders` and `samples.tpch.lineitem`, and a runtime supporting one-to-many Metric View joins (DBR 18.1+ with YAML 1.1). Check the live feature matrix for your compute.

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.metric_views_lab;

## 1. Reproduce the complaint: the header got photocopied

The `orders` table has one header row per order. The `lineitem` table contains several rows per order. A naive joined sum therefore repeats the header total once per line. TPC-H includes both order-level and line-level price fields; they describe different things, so their totals are not expected to match. Run the query and compare the standalone header total with the joined result.

In [ ]:
%sql
SELECT
  (SELECT SUM(o_totalprice) FROM samples.tpch.orders) AS correct_header_total,
  SUM(o.o_totalprice) AS naive_joined_header_total,
  COUNT(DISTINCT o.o_orderkey) AS orders,
  COUNT(*) AS joined_order_line_rows
FROM samples.tpch.orders o
JOIN samples.tpch.lineitem l ON o.o_orderkey = l.l_orderkey;

## 2. Keep the two ledgers at their own grain

Make the order header the primary source and declare `lineitem` as `one_to_many`. The engine can then aggregate the two fact sources independently. An aggregate function must reference columns from one source; arithmetic can combine results from separate aggregations. Fields must remain scalar per primary-source row, so this model does not expose line-item columns as ordinary fields.

In [ ]:
%sql
CREATE OR REPLACE VIEW workspace.metric_views_lab.orders_and_lines_metric_view
WITH METRICS LANGUAGE YAML AS
$$
version: 1.1
comment: "Order headers and line-item facts modeled at separate grains."
source: samples.tpch.orders
joins:
  - name: lineitem
    source: samples.tpch.lineitem
    on: source.o_orderkey = lineitem.l_orderkey
    cardinality: one_to_many
fields:
  - name: order_status
    expr: source.o_orderstatus
  - name: order_date
    expr: source.o_orderdate
measures:
  - name: order_header_revenue
    expr: SUM(source.o_totalprice)
  - name: line_extended_price
    expr: SUM(lineitem.l_extendedprice)
  - name: order_count
    expr: COUNT(DISTINCT source.o_orderkey)
$$;

## 3. Reconcile the minutes with direct controls

These are different economic concepts: order-header total versus line extended price. The meeting's resolution is not that both numbers must match; it is that adding line detail must not multiply header revenue. Compare each measure with a direct query at the same grouping grain.

In [ ]:
%sql
SELECT order_status,
       MEASURE(order_header_revenue) AS header_revenue,
       MEASURE(line_extended_price) AS line_extended_price,
       MEASURE(order_count) AS orders
FROM workspace.metric_views_lab.orders_and_lines_metric_view
GROUP BY order_status
ORDER BY order_status;

In [ ]:
%sql
SELECT o_orderstatus, SUM(o_totalprice) AS direct_header_revenue,
       COUNT(DISTINCT o_orderkey) AS direct_orders
FROM samples.tpch.orders
GROUP BY o_orderstatus
ORDER BY o_orderstatus;

In [ ]:
%sql
SELECT o.o_orderstatus,
       SUM(l.l_extendedprice) AS direct_line_extended_price
FROM samples.tpch.orders o
JOIN samples.tpch.lineitem l ON o.o_orderkey = l.l_orderkey
GROUP BY o.o_orderstatus
ORDER BY o.o_orderstatus;

## 4. What to take back to the office

The chair's ruling: cardinality describes a relationship in the data; it does not turn a guess into a guarantee.

- A `many_to_one` lookup assumes uniqueness on the joined side. Validate the key; do not rely on hope.
- `one_to_many` is a modeling feature, not a repair for arbitrary many-to-many data. Its constraints matter, including field restrictions and materialization/query-rewrite limitations.
- Grain is part of a metric contract. Write it down for every source.
- Build direct SQL controls for expected totals and edge cases before exposing a model broadly.
- If this runtime does not support the feature, pre-model the facts at a clear grain or use a supported model design.

### Exercises
1. Add a customer lookup as a separate many-to-one join and test uniqueness.
2. Compare `COUNT(*)` with distinct order and line counts. Explain the grain behind each value.
3. Design a many-to-many bridge example and decide whether it belongs in this semantic model or an upstream curated table.